# Basic functions

In [5]:
log[z_, branch_ : 0, branchcut_ : -Pi] := 
  Log[Abs[z]] + 
   I*(Arg[z Exp[-I (branchcut + Pi)]] + branchcut + Pi + 
      2 Pi*branch);
arg[z_, branch_ : 0, branchcut_ : -Pi] := 
 Arg[z Exp[-I (branchcut + Pi)]] + branchcut + Pi

F[u_, branch_ : 0, 
   branchcut_ : -Pi] := (-1/(2 Pi*I)) log[(u + I)/(u - I), branch, 
    branchcut];

Fsym[u_, branch_ : 0, branchcut_ : -Pi] := 
  F[u, branch, branchcut] - 1/2;

# The algorithm

In [9]:
uroots[L_, M_, \[Epsilon]Start_, \[Epsilon]Steps_] :=
 Module[{Mset, stringsets, stringsetsPerm, \[Epsilon]list, MM, 
   modes},
  \[Epsilon]list = 
   Range[\[Epsilon]Start, 1, (1 - \[Epsilon]Start)/\[Epsilon]Steps];
  
  Mset = 
   Solve[Sum[s*MM[s], {s, 1, M}] == M, Table[MM[t], {t, 1, M}], 
     NonNegativeIntegers] // Values; (*All possible Ms sets;
  the index is the string length and the value is the number of \
strings of that length; to be called in rootfinder*)
  
  stringsets = 
   Table[Catenate[ 
     Table[Table[i, {j, set[[i]]}], {i, Length[set]}]], {set, 
     Mset}]; (*All possible stringsets; the entries are the strings, 
  with values corresponding to the length of these strings*)
  
  (*stringsetsPerm=Table[Permutations[stringsets[[i]]],{i,Length[
  stringsets]}];All possible stringsets, 
  accounting for the set order of strings corresponding to \
modenumbers increasin as n_1<n_2<...; 
  the sublists are to be called in rootfinder*)
  
  modesforstring[length_, magnon_, stringlength_] := 
   Range[-stringlength*length/2 + (-1)^stringlength*
      stringlength*(magnon - 1)/2, 
    stringlength*length/2 - (-1)^stringlength*
      stringlength*(magnon - 1)/2];
  
  modes = 
   Table[Tuples[modesforstring[L, M, #] & /@ stringsets[[i]]], {i, 
     Length[stringsets]}];
  
  BAElog[length_, magnon_, epsilon_, Ms_, stringlength_, modenumber_, 
    string_] := 
   length*Sum[
      Fsym[2 u[string] + I (stringlength - 1 - 2 k)], {k, 0, 
       stringlength - 1}] - 
    epsilon (-1)^stringlength*
     Sum[Sum[Sum[
        Sum[Fsym[
          u[string] + I (string - 1 - 2 k)/2 - u[j] - 
           I (ss - 1 - 2 kk)/2], {kk, 0, ss - 1}], {j, 1, 
         Ms[[ss]]}], {ss, 1, magnon}], {k, 0, stringlength - 1}] + 
    modenumber;(*Shall change to Fsym modified to account for string \
hypothesis*)
  
  rootfinder[length_, magnon_, epsilon_, Ms_, modeset_, stringvec_, 
    startvalues_] := 
   FindRoot[
    Table[BAElog[length, magnon, epsilon, Ms, stringvec[[k]], 
      modeset[[k]], k], {k, 1, Length[stringvec]}], 
    Table[{u[j], startvalues[[j]]}, {j, 1, Length[stringvec]}]];
  
  ustart[length_, magnon_, stringvec_, 
    modes_] := (1/2) Tan[-Pi*(modes)/L] + 
     0.0000001 /. {ComplexInfinity -> 
      10^15};(*set of the same length as stringsetsPerm[[i,j]]*)
  
  Table[Module[{startvalues}, 
    startvalues = ustart[L, M, stringsets[[i]], modes[[i]][[j]]]; 
    Nest[Module[{}, 
       startvalues = 
        rootfinder[L, M, #, Mset[[i]], modes[[i]][[j]], 
           stringsets[[i]], startvalues] // Values // 
         Re; # + (1 - \[Epsilon]Start)/\[Epsilon]Steps] &, \
\[Epsilon]Start, Length[\[Epsilon]list]]; 
    Table[Chop[startvalues[[s]], 10^-5] + 
      I (stringsets[[i]][[s]] - 1 - 2 k)/2, {s, 
      Length[startvalues]}, {k, 0, stringsets[[i]][[s]] - 1}]], {i, 
    Length[Mset]}, {j, Length[modes[[i]]]}]]

ToExpression::sntx: Invalid syntax in or before "                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

$Failed

In [9]:
A test example:

ToExpression::sntxi: Incomplete expression; more input is needed .


$Failed

In [9]:
answers = uroots[4, 2, 0.1, 100] // Quiet

uroots[4, 2, 0.1, 100]